In [ ]:
import os, sys
import gc
import subprocess
import string
import pickle
import datetime
from tqdm import tqdm
import numpy as np
import xarray as xr
from argparse import ArgumentParser

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path
from matplotlib import gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import calendar
from itertools import chain
import cartopy.mpl.ticker as cticker
import matplotlib.style
import matplotlib as mpl

mpl.style.use('default')

### Directories for the different experiments
- ece3_spinup_dir: The directory with the spinup and evaluation using EC-Earth3 only (the baseline)
- ace2_nemo_spinup_dir: The directory with the spinup using ACE2-NEMO
- ace2_nemo_40yr_eval_dir: The directory containing EC-Earth3 data that is initialised from the 40-year ACE2-NEMO spinup
- ace2_nemo_70yr_eval_dir: The directory containing EC-Earth3 data that is initialised from the 70-year ACE2-NEMO spinup

Also requires a sea mask, which is located in ace2_data_dir.

Figures are saved to the MANUSCRIPT_FIGURE_DIR

In [ ]:
# BASE_DATA_DIR = '/gws/nopw/j04/eerie/cache/bantonio/processed_spinup_data'
BASE_DATA_DIR = '/gws/nopw/j04/iecdt/bantonio/processed_spinup_data'
ece3_spinup_dir = os.path.join(BASE_DATA_DIR, 'EC-Earth3_spinup')
ace2_nemo_40yr_eval_dir = os.path.join(BASE_DATA_DIR, 'ace2-nemo-40yr-spinup-eval')
ace2_nemo_70yr_eval_dir = os.path.join(BASE_DATA_DIR, 'ace2-nemo-70yr-spinup-eval')
ace2_nemo_spinup_dir = os.path.join(BASE_DATA_DIR, 'n3.6_ace2_1951_spinupCMIP6_19510101-20210101')
# ace2_data_dir = '/gws/nopw/j04/eerie/cache/bantonio/ace2_data'
ace2_data_dir = BASE_DATA_DIR
sea_mask = xr.load_dataarray(os.path.join(ace2_data_dir, "era5_sea_mask_ACE2.nc"))

MANUSCRIPT_FIGURE_DIR = '/home/users/bantonio/repos/ace2_nemo_coupler/notebooks/spinup_manuscript_figures'
os.makedirs(MANUSCRIPT_FIGURE_DIR, exist_ok=True)

In [ ]:
def plot_map_grid_cbar_by_row(da_grid,
                                cbar_labels,
                                titles_grid ,
                                vmax_vals,
                                vmin_vals,
                                  projection,
                                  cmaps,
                                width_height_ratio = [8,6],
                                shrink_factor= 1,
                                wspace=0.001,
                                cbar_height_ratio=0.02,
                              lat_ticks=None,
                              lon_ticks=None
                                ):

    num_rows = len(da_grid)
    num_cols = len(da_grid[0])
    
    fig = plt.figure(constrained_layout=True, figsize=(num_cols*shrink_factor*width_height_ratio[0], num_rows*shrink_factor*width_height_ratio[1]))
    
    gs = gridspec.GridSpec(num_rows * 2, 2*num_cols, figure=fig, 
                        width_ratios=[1]* 2*num_cols,
                        height_ratios=[1, 0.02] * num_rows,
                           wspace=wspace) 
    plot_axs = [[fig.add_subplot(gs[2*m, 2*n:2*n+2], projection = projection) for n in range(num_cols)] for m in range(num_rows)]
    
    
    for row in range(num_rows):
        for col in range(num_cols):
            
            plot_da = da_grid[row][col]

            im = plot_da.plot(ax=plot_axs[row][col], 
                              vmax=vmax_vals[row], 
                              vmin=vmin_vals[row], 
                              cmap=cmaps[row], 
                              add_colorbar=False, rasterized=True,
                              transform=ccrs.PlateCarree())
    
            try:
                if lon_ticks is not None:
                    plot_axs[row][col].set_xticks(lon_ticks, crs=ccrs.PlateCarree())
                    lon_formatter = cticker.LongitudeFormatter()
                    plot_axs[row][col].xaxis.set_major_formatter(lon_formatter)
                    plot_axs[row][col].set_xlabel('Longitude')
    
                if col == 0 and lat_ticks is not None:
                    plot_axs[row][col].set_yticks(lat_ticks, crs=ccrs.PlateCarree())
                    lat_formatter = cticker.LatitudeFormatter()
                    plot_axs[row][col].yaxis.set_major_formatter(lat_formatter)
                    plot_axs[row][col].set_ylabel('Latitude')
            except RuntimeError:
                pass
    
            plot_axs[row][col].set_title(titles_grid[row][col])
            plot_axs[row][col].coastlines()
    
        cbar_ax = fig.add_subplot(gs[2*row+1, 1:-1])
        cbar = plt.colorbar(im, cax=cbar_ax, label=cbar_labels[row], orientation='horizontal')
        cbar.ax.tick_params(labelsize=10)
    return fig, plot_axs

def plot_imshow_shared_axes(da_grid, 
                          num_rows, 
                          num_cols, 
                          cbar_label,
                          titles_grid,
                          width_height_ratio = [8,6],
                          shrink_factor=0.7, 
                          wspace=0.001,
                          cbar_height_ratio=0.02,
                          cmap='RdBu_r', 
                          mask=None,
                           **plot_kwargs):
   
    fig = plt.figure(constrained_layout=True, figsize=(shrink_factor*width_height_ratio[0]*2, shrink_factor*width_height_ratio[1]))

    gs = gridspec.GridSpec(num_rows + 1, num_cols, figure=fig, 
                        width_ratios=[1]* num_cols,
                        height_ratios=[1] * num_rows + [0.02],
                           wspace=wspace) 
    plot_axs = [[fig.add_subplot(gs[m, n]) for n in range(num_cols)] for m in range(num_rows)]


    for row in range(num_rows):
        for col in range(num_cols):
            
            plot_da = da_grid[row][col]
            if mask is not None:
                plot_da = xr.where(mask, plot_da, np.nan)
            im = plot_da.plot(ax=plot_axs[row][col], 
                              cmap=cmap, 
                              add_colorbar=False, rasterized=True,
                              **plot_kwargs)

            plot_axs[row][col].set_title(titles_grid[row][col])

    cbar_ax = fig.add_subplot(gs[row+1, :])
    cbar = plt.colorbar(im, cax=cbar_ax, label=cbar_label, orientation='horizontal')
    cbar.ax.tick_params(labelsize=10)

    return fig, plot_axs

In [ ]:
# Handy dict with all the units and abbreviations of variable names

name_lookup = {
           'surface_temperature': {'name':'Surface Temperature', 'units': 'K'},
            'sea_surface_temperature': {'name': 'Sea surface temperature', 'units': 'K', 'abbrev': 'SST'},
               'sea_surface_height': {'name': 'Sea Surface Height', 'units': 'm', 'abbrev': 'SSH'},
               'mixed_layer_depth': {'name': 'Mixed Layer Depth', 'units': 'm', 'abbrev': 'MLD'},
              'sea_ice_fraction': {'name':'Sea Ice Fraction', 'units': 'Fraction', 'abbrev': 'siconc'},
              'sea_ice_thickness': {'name':'Sea Ice Thickness', 'units': 'm', 'abbrev': 'SIthick'},
               'sea_ice_extent': {'name':'Sea Ice Extent', 'units': '$km^2$', 'abbrev': 'SIext'},
              'sea_ice_volume': {'name':'Sea Ice Volume', 'units': '$m^3$', 'abbrev': 'SIvol'},
              'LHTFLsfc': {'name':'Latent heat flux', 'units': '$W/m^2$'}, 
               'SHTFLsfc': {'name':'Sensible heat flux', 'units': '$W/m^2$'}, 
               'DLWRFsfc': {'name':'LW flux down', 'units': '$W/m^2$'}, 
               'ULWRFsfc': {'name':'LW flux up', 'units': '$W/m^2$'},
               'DSWRFsfc': {'name':'SW flux down', 'units': '$W/m^2$'},
               'USWRFsfc': {'name':'SW flux up', 'units': '$W/m^2$'},
               'PRATEsfc': {'name':'Precipitation rate', 'units': '$kg/m^2/s$', 'abbrev': 'TP'},
                'total_precipitation': {'name':'Precipitation rate', 'units': '$kg/m^2/s$', 'abbrev': 'TP'},
               'total_precipitation_daily': {'name':'Precipitation', 'units': 'mm/day', 'abbrev': 'P'},
                'TMP2m': {'name': '2-metre temperature', 'units': '$K$', 'abbrev': 'T2m'},
              '2m_temperature': {'name': '2-metre temperature', 'units': '$K$', 'abbrev': 'T2m'},
               '2m_temperature_sea_points': {'name': '2-metre temperature (sea points)', 'units': '$K$', 'abbrev': 'T2m sea'},
              'mean_surface_sensible_heat_flux': {'name': 'Sensible heat flux', 'units': '$W/m^2$', 'abbrev': 'SHF'},
              'mean_surface_latent_heat_flux': {'name': 'Latent heat flux', 'units': '$W/m^2$', 'abbrev': 'LHF'},
               'sea_water_potential_temperature': {'name': 'Sea water potential temperature', 'units': '$K$', 'abbrev': r"$\theta_o$$"},
               'mean_surface_upward_long_wave_radiation_flux': {'name': 'Upward LW Radiation Flux',  'units': '$W/m^2$'},
               'mean_surface_downward_long_wave_radiation_flux': {'name': 'Downward LW Radiation Flux', 'units': '$W/m^2$'},
               'mean_surface_upward_short_wave_radiation_flux': {'name': 'Upward SW Radiation Flux', 'abbrev':r'$R_{sw\uparrow}$','units': '$W/m^2$'},
               'mean_surface_downward_short_wave_radiation_flux': {'name': 'Downward SW Radiation Flux', 'abbrev':r'$R_{sw\downarrow}$','units': '$W/m^2$'},
            'mean_surface_net_short_wave_radiation_flux': {'name': 'Net SW Radiation Flux','abbrev': r'$R_{sw,net}$', 'units': '$W/m^2$'},
               'mean_surface_upward_short_wave_radiation_flux_oce': {'name': 'Upward SW Radiation Flux (ocean)', 'abbrev':r'$R_{sw\uparrow}$','units': '$W/m^2$'},
               'mean_surface_downward_short_wave_radiation_flux_oce': {'name': 'Downward SW Radiation Flux (ocean)', 'abbrev':r'$R_{sw\downarrow}$','units': '$W/m^2$'},
            'mean_surface_net_short_wave_radiation_flux_oce': {'name': 'Net SW Radiation Flux (ocean)','abbrev': r'$R_{sw,net}$', 'units': '$W/m^2$'},
                'mean_surface_upward_short_wave_radiation_flux_ice': {'name': 'Upward SW Radiation Flux (ice)', 'abbrev':r'$R_{sw\uparrow}$','units': '$W/m^2$'},
               'mean_surface_downward_short_wave_radiation_flux_ice': {'name': 'Downward SW Radiation Flux (ice)', 'abbrev':r'$R_{sw\downarrow}$','units': '$W/m^2$'},
            'mean_surface_net_short_wave_radiation_flux_ice': {'name': 'Net SW Radiation Flux (ice)','abbrev': r'$R_{sw,net}$', 'units': '$W/m^2$'},
               'mean_surface_net_long_wave_radiation_flux': {'name': 'Net LW Radiation Flux', 'abbrev':r'$R_{lw,net}$', 'units': '$W/m^2$'},
               'heat_content': {'name': 'Ocean heat content', 'abbrev': 'OHC', 'units': '$J/m^2$'},
               'surface_temperature_difference': {'name': 'T2m - Sea Ice Temperature', 'units': '$K$'},
               'total_water_path': {'name': 'Total water path', 'abbrev': 'TWP', 'units': '$mm$'},
               'albedo_oce': {'name': 'Albedo (ocean)', 'abbrev': r'$\alpha_{oce}$', 'units': 'Fraction'},
               'albedo_ice': {'name': 'Albedo (ice)', 'abbrev': r'$\alpha_{ice}$', 'units': 'Fraction'},
               '10m_u_component_of_wind': {'name': '10m eastward wind', 'units': '$m s^{-1}$'},
               'instantaneous_eastward_turbulent_surface_stress': {'name': 'Eastward wind stress', 'units': '$Nm^{-2}$'},
                'instantaneous_northward_turbulent_surface_stress': {'name': 'Northward wind stress', 'units': '$Nm^{-2}$'},
                'total_heat_flux': {'name': 'Total heat flux', 'units': '$Wm^{-2}$', 'abbrev': 'Total HF'},
               'total_heat_flux_oce': {'name': 'Total heat flux (ocean)', 'units': '$Wm^{-2}$', 'abbrev': 'Total HF (oce)'},
               'mean_surface_heat_flux': {'name': 'Latent + sensible heat flux', 'units': '$Wm^{-2}$', 'abbrev': 'LHF + SHF'},
                'sea_surface_salinity': {'name': 'Sea surface salinity', 'units': r'$10^{-3} ppt$', 'abbrev': 'SSS'}
}

## Plots of aggregated evolution in time

These are plots of the data aggregated over some spatial domain

For each processed dataset, we load the mean_dict.pkl object. 
This is a nested dict; the first layer of keys determines the spatial domain (e.g. global, northern hemisphere). The second layer determines the kind of aggregation; this will be mean for most variables, but for sea ice volume it is an unweighted sum.

So e.g. globally averaged time series data for the ECE3 spinup can be extracted from ece_spinup_mean_dict['Global']['mean']

Note that the ECE3 runs that start from ACE2-NEMO spinups have dates that start from 1951, so we have to convert the time if we want to combine it with the spinup data

In [ ]:
with open(os.path.join(ece3_spinup_dir, f'mean_dict.pkl'), 'rb') as ifh:
    ece_spinup_mean_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_40yr_eval_dir, f'mean_dict.pkl'), 'rb') as ifh:
    ace2_nemo_40yr_eval_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_70yr_eval_dir, f'mean_dict.pkl'), 'rb') as ifh:
    ace2_nemo_70yr_eval_dict = pickle.load(ifh)
    
with open(os.path.join(ace2_nemo_spinup_dir, f'mean_dict.pkl'), 'rb') as ifh:
    ace2_nemo_spinup_dict = pickle.load(ifh)


In [ ]:
ace2_nemo_40yr_eval_time =pd.date_range('1991-01-01', '2061-01-01', freq='MS')[:len(ace2_nemo_40yr_eval_dict['Global']['mean']['time'])]
ace2_nemo_70yr_eval_time =pd.date_range('2021-01-01', '2081-01-01', freq='MS')[:len(ace2_nemo_70yr_eval_dict['Global']['mean']['time'])]

The plot below just compares the spinups by themselves, without any evaluation

In [ ]:
vars_to_plot = [
                'sea_surface_temperature',  'sea_ice_volume', 'sea_surface_height', 'sea_surface_salinity']
area_name = 'Global'
ncols=2
nrows = int(np.ceil(len(vars_to_plot)/ncols))

fig, axs = plt.subplots(nrows, ncols, figsize=(ncols*6, 4*nrows))
fig.tight_layout(pad=5)
handles = []
labels = []

time_vals = ece_spinup_mean_dict[area_name]['mean']['time']
for n, var in enumerate(vars_to_plot):

    row = int(n/ncols)
    col = n%ncols

    label = 'ECE3 spinup'

    if var == 'sea_ice_volume':
        aggregation='UnweightedSum'
    else:
        aggregation='mean'
    ece_time_series = ece_spinup_mean_dict[area_name][aggregation][var].groupby('time.year').mean().sel(year=slice(1951,2020))
        
    h = (ece_time_series ).plot(ax=axs[row,col], label=label)    

    if n ==0:
        handles.append(h[0])
        labels.append(label)

    label='ACE2-NEMO-spinup'

    ace2_nemo_spinup_time_series = ace2_nemo_spinup_dict[area_name][aggregation][var].groupby('time.year').mean().sel(year=slice(1951,2020))

    h = ace2_nemo_spinup_time_series.plot(ax=axs[row,col], label=label)    
    if n ==0:
        handles.append(h[0])
        labels.append(label)

    axs[row,col].set_title(f"{var}")
    axs[row,col].set_ylabel(f"{name_lookup[var]['abbrev']} [{name_lookup[var]['units']}]")
    axs[row,col].set_title(f"({string.ascii_lowercase[n]}) {name_lookup[var]['name']}")
    axs[row,col].set_xlabel('Time')

fig.subplots_adjust(bottom=0.3, wspace=0.33)
axs[-1,-1].legend(handles = handles , labels=labels,loc='upper center', 
             bbox_to_anchor=(-0.3, -0.2),fancybox=False, shadow=False, ncol=4)
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, 'spinup_comparison.pdf'), format='pdf', bbox_inches='tight')

Investigate the transition point between the end of the spin-up run, and the start of the ECE3 evaluation run

Note that I found a bug in the postprocessing of the ACE2-NEMO data, that means it cuts short the postprocessing at the very end of the decade (at the point it creates restarts). But only for some of the data, not the data saved by NEMO. So sea ice volume is 0 here because of this missing data (I do have sea ice concentration as output from NEMO, but currently my pipeline isn't setup to use it)

In [ ]:
vars_to_plot = [
                'sea_surface_temperature',  'sea_ice_volume', 'sea_surface_height', 'sea_surface_salinity']
area_name = 'Global'
ncols=2
nrows = int(np.ceil(len(vars_to_plot)/ncols))

fig, axs = plt.subplots(nrows, ncols, figsize=(ncols*6, 4*nrows))
fig.tight_layout(pad=5)
handles = []
labels = []

time_vals = ece_spinup_mean_dict[area_name]['mean']['time']
for n, var in enumerate(vars_to_plot):

    row = int(n/ncols)
    col = n%ncols

    label = 'ECE3_spinup'

    if var == 'sea_ice_volume':
        aggregation='UnweightedSum'
    else:
        aggregation='mean'
    ece_time_series = ece_spinup_mean_dict[area_name][aggregation][var]
        
    h = (ece_time_series ).sel(time=slice(datetime.datetime(1990,6,1),datetime.datetime(1991,6,1))).plot(ax=axs[row,col], label=label)    

    if n ==0:
        handles.append(h[0])
        labels.append(label)


    label='ACE2-NEMO-40yr'
    ace2_nemo_eval_time_series_raw = ace2_nemo_40yr_eval_dict[area_name][aggregation][var].isel(time=range(len(ace2_nemo_40yr_eval_time)))
    ace2_nemo_eval_time_series_raw = ace2_nemo_eval_time_series_raw.assign_coords({'time': ace2_nemo_40yr_eval_time})
    ace2_nemo_eval_time_series = ace2_nemo_eval_time_series_raw
    
    ace2_nemo_spinup_time_series = ace2_nemo_spinup_dict[area_name][aggregation][var].sel(time=slice(datetime.datetime(1951,1,1),datetime.datetime(1990,12,20)))

    ace2_nemo_time_series = xr.concat([ace2_nemo_spinup_time_series, ace2_nemo_eval_time_series], dim='time')
        
    h = ace2_nemo_time_series.sel(time=slice(datetime.datetime(1990,6,1),datetime.datetime(1991,6,1))).plot(ax=axs[row,col], label=label)    
    if n ==0:
        handles.append(h[0])
        labels.append(label)

    axs[row,col].set_title(f"{var}")
    axs[row,col].set_ylabel(f"{name_lookup[var]['abbrev']} [{name_lookup[var]['units']}]")
    axs[row,col].set_title(f"({string.ascii_lowercase[n]}) {name_lookup[var]['name']}")
    axs[row,col].set_xlabel('Time')

fig.subplots_adjust(bottom=0.3, wspace=0.33)
axs[-1,-1].legend(handles = handles , labels=labels,loc='upper center', 
             bbox_to_anchor=(-0.3, -0.2),fancybox=False, shadow=False, ncol=4)

# plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, 'spinup_and_eval_comparison.pdf'), format='pdf', bbox_inches='tight')


Investigate the transition point between the end of the spin-up run, and the start of the ECE3 evaluation run, for the 70-year spinup

Note that I found a bug in the postprocessing of the ACE2-NEMO data, that means it cuts short the postprocessing at the very end of the decade (at the point it creates restarts). But only for some of the data, not the data saved by NEMO. So sea ice volume is 0 here because of this missing data (I do have sea ice concentration as output from NEMO, but currently my pipeline isn't setup to use it)

In [ ]:
vars_to_plot = [
                'sea_surface_temperature',  'sea_ice_volume', 'sea_surface_height', 'sea_surface_salinity']
area_name = 'Global'
ncols=2
nrows = int(np.ceil(len(vars_to_plot)/ncols))

fig, axs = plt.subplots(nrows, ncols, figsize=(ncols*6, 4*nrows))
fig.tight_layout(pad=5)
handles = []
labels = []

time_vals = ece_spinup_mean_dict[area_name]['mean']['time']
for n, var in enumerate(vars_to_plot):

    row = int(n/ncols)
    col = n%ncols

    label = 'ECE3_spinup'

    if var == 'sea_ice_volume':
        aggregation='UnweightedSum'
    else:
        aggregation='mean'
    ece_time_series = ece_spinup_mean_dict[area_name][aggregation][var]
        
    h = (ece_time_series ).sel(time=slice(datetime.datetime(2020,6,1),datetime.datetime(2021,6,1))).plot(ax=axs[row,col], label=label)    

    if n ==0:
        handles.append(h[0])
        labels.append(label)


    label='ACE2-NEMO-70yr'
    ace2_nemo_eval_time_series_raw = ace2_nemo_70yr_eval_dict[area_name][aggregation][var].isel(time=range(len(ace2_nemo_70yr_eval_time)))
    ace2_nemo_eval_time_series_raw = ace2_nemo_eval_time_series_raw.assign_coords({'time': ace2_nemo_70yr_eval_time})
    ace2_nemo_eval_time_series = ace2_nemo_eval_time_series_raw
    
    ace2_nemo_spinup_time_series = ace2_nemo_spinup_dict[area_name][aggregation][var].sel(time=slice(datetime.datetime(1951,1,1),datetime.datetime(2020,12,1)))

    ace2_nemo_time_series = xr.concat([ace2_nemo_spinup_time_series, ace2_nemo_eval_time_series], dim='time')
        
    h = ace2_nemo_time_series.sel(time=slice(datetime.datetime(2020,6,1),datetime.datetime(2021,6,1))).plot(ax=axs[row,col], label=label)    
    if n ==0:
        handles.append(h[0])
        labels.append(label)

    axs[row,col].set_title(f"{var}")
    axs[row,col].set_ylabel(f"{name_lookup[var]['abbrev']} [{name_lookup[var]['units']}]")
    axs[row,col].set_title(f"({string.ascii_lowercase[n]}) {name_lookup[var]['name']}")
    axs[row,col].set_xlabel('Time')

fig.subplots_adjust(bottom=0.3, wspace=0.33)
axs[-1,-1].legend(handles = handles , labels=labels,loc='upper center', 
             bbox_to_anchor=(-0.3, -0.2),fancybox=False, shadow=False, ncol=4)

# plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, 'spinup_and_eval_comparison.pdf'), format='pdf', bbox_inches='tight')


The plot below stitches the ACE2-NEMO spinup with the ECE3 run that is initialised from the spun-up ocean

In [ ]:
vars_to_plot = [
                'sea_surface_temperature',  'sea_ice_volume', 'sea_surface_height', 'sea_surface_salinity']
area_name = 'Global'
ncols=2
nrows = int(np.ceil(len(vars_to_plot)/ncols))

fig, axs = plt.subplots(nrows, ncols, figsize=(ncols*6, 4*nrows))
fig.tight_layout(pad=5)
handles = []
labels = []

time_vals = ece_spinup_mean_dict[area_name]['mean']['time']
for n, var in enumerate(vars_to_plot):

    row = int(n/ncols)
    col = n%ncols

    label = 'ECE3_spinup'

    if var == 'sea_ice_volume':
        aggregation='UnweightedSum'
    else:
        aggregation='mean'
    ece_time_series = ece_spinup_mean_dict[area_name][aggregation][var].groupby('time.year').mean()
        
    h = (ece_time_series ).plot(ax=axs[row,col], label=label)    

    if n ==0:
        handles.append(h[0])
        labels.append(label)

    label='ACE2-NEMO-eval'
    ace2_nemo_eval_time_series_raw = ace2_nemo_40yr_eval_dict[area_name][aggregation][var].isel(time=range(len(ace2_nemo_40yr_eval_time)))
    ace2_nemo_eval_time_series_raw = ace2_nemo_eval_time_series_raw.assign_coords({'time': ace2_nemo_40yr_eval_time})
    ace2_nemo_eval_time_series = ace2_nemo_eval_time_series_raw.groupby('time.year').mean()

    h = ace2_nemo_eval_time_series.plot(ax=axs[row,col], label=label)   
    if n ==0:
        handles.append(h[0])
        labels.append(label)

    label='ACE2-NEMO-spinup'
    ace2_nemo_spinup_time_series = ace2_nemo_spinup_dict[area_name][aggregation][var].groupby('time.year').mean().sel(year=slice(1951,1991))
    h = ace2_nemo_spinup_time_series.plot(ax=axs[row,col], label=label)
    # ace2_nemo_spinup_time_series
    # ace2_nemo_time_series = xr.concat([ace2_nemo_spinup_time_series, ace2_nemo_eval_time_series], dim='year')
        
    # h = ace2_nemo_time_series.plot(ax=axs[row,col], label=label)    
    if n ==0:
        handles.append(h[0])
        labels.append(label)

    axs[row,col].set_title(f"{var}")
    axs[row,col].set_ylabel(f"{name_lookup[var]['abbrev']} [{name_lookup[var]['units']}]")
    axs[row,col].set_title(f"({string.ascii_lowercase[n]}) {name_lookup[var]['name']}")
    axs[row,col].set_xlabel('Time')

fig.subplots_adjust(bottom=0.3, wspace=0.33)
axs[-1,-1].legend(handles = handles , labels=labels,loc='upper center', 
             bbox_to_anchor=(-0.3, -0.2),fancybox=False, shadow=False, ncol=4)

plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, 'spinup_and_eval_comparison.pdf'), format='pdf', bbox_inches='tight')


# Hovmoller plot

In [ ]:
with open(os.path.join(ece3_spinup_dir, f'mean_dict.pkl'), 'rb') as ifh:
    ece_spinup_mean_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_40yr_eval_dir, f'mean_dict.pkl'), 'rb') as ifh:
    ace2_nemo_40yr_eval_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_spinup_dir, f'mean_dict.pkl'), 'rb') as ifh:
    ace2_nemo_spinup_dict = pickle.load(ifh)


In [ ]:
max_level=3000
pot_temp_da = ace2_nemo_spinup_dict['Global']['mean']['sea_water_potential_temperature'].groupby('time.year').mean().sel(member=0).sel(olevel=slice(0,max_level)).transpose('olevel', 'year')
pot_temp_da_ece = ece_spinup_mean_dict['Global']['mean']['sea_water_potential_temperature'].groupby('time.year').mean().sel(olevel=slice(0,max_level)).transpose('olevel', 'year')

da_dict = {'Baseline': pot_temp_da_ece- pot_temp_da_ece.sel(year=1951),
           'ACE2-NEMO': pot_temp_da- pot_temp_da.sel(year=1951)
          }

fig, axs = plot_imshow_shared_axes(da_grid= [list(da_dict.values())], 
                              num_rows=1, 
                              num_cols=len(da_dict.keys()), 
                              titles_grid=[["" for item in da_dict.keys()]],
                              cbar_label=f"Potential temperature change [K]",
                              shrink_factor=0.7, 
                              cmap='RdBu_r', 
                              mask=None,
                              vmin=-0.4, 
                              vmax=0.4,
                              yincrease=False)

axs[0][0].set_title('(a) Baseline')
axs[0][1].set_title('(b) ACE2-NEMO')

for a in axs[0]:
    a.set_xlabel('Year')
    a.set_ylabel('Depth [m]')
    
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f'ocean_hovmoller_spinup.pdf'), format='pdf')

Hovmoller plot stitching together the 40-year ACE2-NEMO spinup and the ECE3 eval period initialised from that spinup

In [ ]:
max_level=3000
pot_temp_spinup_da = ace2_nemo_spinup_dict['Global']['mean']['sea_water_potential_temperature'].groupby('time.year').mean().sel(member=0).sel(olevel=slice(0,max_level)).transpose('olevel', 'year')
pot_temp_da_ece = ece_spinup_mean_dict['Global']['mean']['sea_water_potential_temperature'].groupby('time.year').mean().sel(olevel=slice(0,max_level)).transpose('olevel', 'year')
pot_temp_eval_da = ace2_nemo_40yr_eval_dict['Global']['mean']['sea_water_potential_temperature'].groupby('time.year').mean().sel(olevel=slice(0,max_level)).transpose('olevel', 'year')

pot_temp_full = xr.concat([pot_temp_spinup_da.sel(year=slice(1951,1991)), pot_temp_eval_da.sel(year=slice(1992,2050))], dim='year')

da_dict = {'Baseline': pot_temp_da_ece- pot_temp_da_ece.sel(year=1951),
           'ACE2-NEMO': pot_temp_full- pot_temp_full.sel(year=1951)
          }

fig, axs = plot_imshow_shared_axes(da_grid= [list(da_dict.values())], 
                              num_rows=1, 
                              num_cols=len(da_dict.keys()), 
                              titles_grid=[["" for item in da_dict.keys()]],
                              cbar_label=f"Potential temperature change [K]",
                              shrink_factor=0.7, 
                              cmap='RdBu_r', 
                              mask=None,
                              vmin=-0.4, 
                              vmax=0.4,
                              yincrease=False)

axs[0][0].set_title('(a) Baseline')
axs[0][1].set_title('(b) ACE2-NEMO')

for a in axs[0]:
    a.set_xlabel('Year')
    a.set_ylabel('Depth [m]')
    
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f'ocean_hovmoller_full.pdf'), format='pdf')

# Global averages of temperatures at depth

The processed data has ocean potential temperature (theta) averaged in different depth bins.
This variable is called thetao_binned in the EC-Earth data, and toce_pot_binned in the ACE2-NEMO data


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(6*2,4))
fig.tight_layout(pad=5)
handles = []

thetao_da = ece_spinup_mean_dict['Global']['mean']['thetao_binned']
thetao_ace2_nemo_da = ace2_nemo_spinup_dict['Global']['mean']['toce_pot_binned']

for lb in thetao_da['lev_bins'].values:
    tmp_da = thetao_da.sel(lev_bins=lb)
    tmp_da = (tmp_da -tmp_da.isel(time=0))/ tmp_da.std('time')
    h = tmp_da.plot(ax=ax[0], label=lb + ' m')
    handles.append(h[0])

for lb in thetao_ace2_nemo_da['olevel_bins'].values:
    tmp_da = thetao_ace2_nemo_da.sel(olevel_bins=lb)
    tmp_da = (tmp_da -tmp_da.isel(time=0))/ tmp_da.std('time')
    h = tmp_da.plot(ax=ax[1], label=lb + ' m')

model_labels = ['Baseline','ACE2-NEMO']
for n, a in enumerate(ax):
    a.set_title(f'({string.ascii_lowercase[n]}) {model_labels[n]} \n Potential temperature change')
    a.set_ylabel(r'$\theta-\theta_{t=0}$ [K]')
    a.set_xlabel('Year')
    a.set_ylim([-6, 6])

ax[1].legend(handles = handles , labels=[v + 'm' for v  in thetao_ace2_nemo_da['olevel_bins'].values], loc='upper center', 
             bbox_to_anchor=(-0.2, -0.2),fancybox=False, shadow=False, ncol=len(thetao_ace2_nemo_da['olevel_bins'].values))
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, 'thetao_spinup_comparison.pdf'), format='pdf', bbox_inches='tight')

# Trends in potential temperature averaged by latitude

polyfit_toce_latitude_dict contains linear regressions fitted to ocean potential temperature data that has first been averaged by latitude first.
The trend is calculated over several different time ranges; the range is determined by which key of the dict you specify. e.g. polyfit_toce_latitude_dict['last 20 years'] is a linear regresion fitted over the last 20 years 

In [ ]:
with open(os.path.join(ece3_spinup_dir, f'polyfit_toce_latitude_dict.pkl'), 'rb') as ifh:
    ece_spinup_toce_trend_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_40yr_eval_dir, f'polyfit_toce_latitude_dict.pkl'), 'rb') as ifh:
    ace2_nemo_40yr_eval_toce_trend_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_spinup_dir, f'polyfit_toce_latitude_dict.pkl'), 'rb') as ifh:
    ace2_nemo_spinup_toce_trend_dict = pickle.load(ifh)

In [ ]:
# Trend in ocean potential temperature for the last 20 years of the ECE3 spinup and the eval period of the ACE2-NEMO spinup

mpl.style.use('default')
drift_vars = ['toce_latitude']
time_period = 'last 20 years'

da_grid = [ [ece_spinup_toce_trend_dict[time_period]['polyfit_coefficients'].sel(degree=1).transpose('olevel', 'latitude'),
             ace2_nemo_40yr_eval_toce_trend_dict[time_period]['polyfit_coefficients'].sel(degree=1).transpose('olevel', 'latitude')]]

num_cols = len(da_grid[0])
num_rows = len(da_grid)
cbar_labels= [f" drift /year" for varname in drift_vars]
titles_grid = [[f'a) Baseline ({time_period.title()})', f'b) ACE2-NEMO ({time_period.title()})']]

vmax_vals = [0.02]
vmin_vals = [-1*item for item in vmax_vals]

width_height_ratio = [8,5]
shrink_factor=0.7
wspace=0.001
cbar_height_ratio=0.02
cmap='RdBu_r'
mask=None

fig = plt.figure(constrained_layout=True, figsize=(shrink_factor*width_height_ratio[0]*2, shrink_factor*width_height_ratio[1]))

gs = gridspec.GridSpec(num_rows + 1, num_cols, figure=fig, 
                    width_ratios=[1]* num_cols,
                    height_ratios=[1] * num_rows + [0.02],
                       wspace=wspace) 
plot_axs = [[fig.add_subplot(gs[m, n]) for n in range(num_cols)] for m in range(num_rows)]


for col in range(num_cols):
    im0 = da_grid[0][col].sortby('olevel', ascending=False).plot(ax=plot_axs[0][col], 
                      cmap=cmap, 
                      add_colorbar=False, 
                      rasterized=True,
                               vmin=-0.04,
                               vmax=0.04,
                            yincrease=False
                           )
    plot_axs[0][col].set_title(titles_grid[0][col])
    
    if col == 0:
        plot_axs[0][col].set_ylabel('Ocean level')
    else:
        plot_axs[0][col].set_yticks([])
        plot_axs[0][col].set_ylabel('')
    plot_axs[0][col].set_xlim([-80, 90])

    lat_formatter = cticker.LatitudeFormatter()
    plot_axs[0][col].xaxis.set_major_formatter(lat_formatter)
    plot_axs[0][col].set_xlabel('Latitude')
    
cbar_ax = fig.add_subplot(gs[1, :])
cbar = plt.colorbar(im0, cax=cbar_ax, label='Potential temperature drift [K/year]', orientation='horizontal')
cbar.ax.tick_params(labelsize=10)
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f'toce_latitude_drift_eval_{time_period.replace(' ', '-')}.pdf'), format='pdf')

In [ ]:
# Trend in ocean potential temperature at the end of the spinup period (1980-1990)

mpl.style.use('default')
drift_vars = ['toce_latitude']
time_period = '1980-1990'

da_grid = [ [ece_spinup_toce_trend_dict[time_period]['polyfit_coefficients'].sel(degree=1).transpose('olevel', 'latitude'),
             ace2_nemo_spinup_toce_trend_dict[time_period]['polyfit_coefficients'].sel(degree=1).transpose('olevel', 'latitude')]]

num_cols = len(da_grid[0])
num_rows = len(da_grid)
cbar_labels= [f" drift /year" for varname in drift_vars]
titles_grid = [[f'a) Baseline ({time_period.title()})', f'b) ACE2-NEMO ({time_period.title()})']]

vmax_vals = [0.04]
vmin_vals = [-1*item for item in vmax_vals]

width_height_ratio = [8,5]
shrink_factor=0.7
wspace=0.001
cbar_height_ratio=0.02
cmap='RdBu_r'
mask=None

fig = plt.figure(constrained_layout=True, figsize=(shrink_factor*width_height_ratio[0]*2, shrink_factor*width_height_ratio[1]))

gs = gridspec.GridSpec(num_rows + 1, num_cols, figure=fig, 
                    width_ratios=[1]* num_cols,
                    height_ratios=[1] * num_rows + [0.02],
                       wspace=wspace) 
plot_axs = [[fig.add_subplot(gs[m, n]) for n in range(num_cols)] for m in range(num_rows)]


for col in range(num_cols):
    im0 = da_grid[0][col].sortby('olevel', ascending=False).plot(ax=plot_axs[0][col], 
                      cmap=cmap, 
                      add_colorbar=False, 
                      rasterized=True,
                               vmin=-0.06,
                               vmax=0.06,
                            yincrease=False
                           )
    plot_axs[0][col].set_title(titles_grid[0][col])
    
    if col == 0:
        plot_axs[0][col].set_ylabel('Ocean level')
    else:
        plot_axs[0][col].set_yticks([])
        plot_axs[0][col].set_ylabel('')
    plot_axs[0][col].set_xlim([-80, 90])

    lat_formatter = cticker.LatitudeFormatter()
    plot_axs[0][col].xaxis.set_major_formatter(lat_formatter)
    plot_axs[0][col].set_xlabel('Latitude')
    
cbar_ax = fig.add_subplot(gs[1, :])
cbar = plt.colorbar(im0, cax=cbar_ax, label='Potential temperature drift [K/year]', orientation='horizontal')
cbar.ax.tick_params(labelsize=10)
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f'toce_latitude_drift_spinup_{time_period.replace(' ', '-')}.pdf'), format='pdf')

# Maps of surface trends

trends_dict contains linear regressions fitted to surface variables.
The trend is calculated over several different time ranges; the range is determined by which key of the dict you specify. e.g. polyfit_toce_latitude_dict['1980-1990'] is a linear regresion fitted over the last 20 years.

We need to make sure that the different time ranges used are consistent. If comparing spin-ups, then best to use the date ranges (e.g. 1980-19990). But if comparing the ACE2-NEMO spinup evaluation dataset with the baseline spinup dataset, you can use 'last 20 years' or similar.

If you need a different range, then it is easy to modify notebooks/process_ece_data.ipynb to add diferent ranges for the ECE3 data. A bit more complicated for the ACE2-NEMO data, as the data is on ECMWF cluster

In [ ]:
with open(os.path.join(ece3_spinup_dir, f'trends_dict.pkl'), 'rb') as ifh:
    ece_spinup_trend_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_40yr_eval_dir, f'trends_dict.pkl'), 'rb') as ifh:
    ace2_nemo_40yr_eval_trend_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_spinup_dir, f'trends_dict.pkl'), 'rb') as ifh:
    ace2_nemo_spinup_trend_dict = pickle.load(ifh)

In [ ]:
# Surface trends for the last 10 years of the 40-year spin up periods

mpl.style.use('default')
drift_vars = ['sea_surface_temperature',  'sea_surface_height']
time_period = '1980-1990'
da_grid = [ [ece_spinup_trend_dict[time_period][varname]['polyfit_coefficients'].sel(degree=1).transpose('latitude', 'longitude'),
             ace2_nemo_spinup_trend_dict[time_period][varname]['polyfit_coefficients'].sel(degree=1).transpose('latitude', 'longitude')] for varname in drift_vars]

num_cols = 2
num_rows = 4
cbar_labels= [f"{name_lookup[varname]['name']} trend [{name_lookup[varname]['units']}/year]" for varname in drift_vars]
titles_grid = [[f'a) Baseline ({time_period.title()})', f'b) ACE2-NEMO ({time_period.title()})'],
               [f'c) Baseline ({time_period.title()})', f'd) ACE2-NEMO ({time_period.title()})']]
# vmax_vals = [mean_state_range_dict.get(varname, {}).get('vmax', None) for varname in drift_vars]
# vmin_vals = [mean_state_range_dict.get(varname, {}).get('vmin', None) for varname in drift_vars]
vmax_vals = [0.2, 0.02]
vmin_vals = [-1*item for item in vmax_vals]
cmaps = ['RdBu_r']*2

plot_map_grid_cbar_by_row(da_grid,
                                cbar_labels,
                                titles_grid ,
                                vmax_vals,
                                vmin_vals,
                                  projection=ccrs.Robinson(central_longitude=180),
                                  cmaps=cmaps,
                                width_height_ratio = [8,6],
                                shrink_factor= 0.7,
                                wspace=0.001,
                                cbar_height_ratio=0.02,
                                )
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f'surface_drifts_spinup_{time_period.replace(' ', '-')}.pdf'), format='pdf')

In [ ]:
# Surface trends for the 50 year eval periods

mpl.style.use('default')
drift_vars = ['sea_surface_temperature',  'sea_surface_height']
time_period = 'last 20 years'
da_grid = [ [ece_spinup_trend_dict[time_period][varname]['polyfit_coefficients'].sel(degree=1).transpose('latitude', 'longitude'),
             ace2_nemo_40yr_eval_trend_dict[time_period][varname]['polyfit_coefficients'].sel(degree=1).transpose('latitude', 'longitude')] for varname in drift_vars]
num_cols = 2
num_rows = 4
cbar_labels= [f"{name_lookup[varname]['name']} trend [{name_lookup[varname]['units']}/year]" for varname in drift_vars]
titles_grid = [[f'a) Baseline ({time_period.title()})', f'b) ACE2-NEMO ({time_period.title()})'],
               [f'c) Baseline ({time_period.title()})', f'd) ACE2-NEMO ({time_period.title()})']]
# vmax_vals = [mean_state_range_dict.get(varname, {}).get('vmax', None) for varname in drift_vars]
# vmin_vals = [mean_state_range_dict.get(varname, {}).get('vmin', None) for varname in drift_vars]
vmax_vals = [0.1, 0.01]
vmin_vals = [-1*item for item in vmax_vals]
cmaps = ['RdBu_r']*2

plot_map_grid_cbar_by_row(da_grid,
                                cbar_labels,
                                titles_grid ,
                                vmax_vals,
                                vmin_vals,
                                  projection=ccrs.Robinson(central_longitude=180),
                                  cmaps=cmaps,
                                width_height_ratio = [8,6],
                                shrink_factor= 0.7,
                                wspace=0.001,
                                cbar_height_ratio=0.02,
                                )
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f'surface_drifts_eval_{time_period.replace(' ', '-')}.pdf'), format='pdf')

# mean state comparison

time_mean_state_dict.pkl is an average of the variables over a particular time period. 'All' just means over the full time range. Other options include:

['Pre-1980', 'Post-1980', 'All January', 'JJA', 'DJF', '1st month', '1st year', '5th year', '1st decade', 'last 50 years', 'last 20 years', 'last 10 years', '1980-1990', '1970-1990', '1980-2000', '1990-2000', '2000-2020', '2010-2020', 'All']

In [ ]:
mean_state_range_dict = {'total_precipitation_daily': {'vmin': 0, 'vmax': 20, 'cmap': 'Blues'}, 
                         'sea_surface_temperature': {'vmin': 270, 'vmax': 305},
                         '2m_temperature': {'vmin': 225, 'vmax': 305}, 
                         'sea_surface_height': {'vmin': -2, 'vmax': 2},
                          'mean_surface_sensible_heat_flux': {'vmin': -100, 'vmax': 100, 'cmap': 'RdBu_r'},
                         'mean_surface_latent_heat_flux': {'vmin': -200, 'vmax': 200, 'cmap': 'RdBu_r'},
                        'mean_surface_net_long_wave_radiation_flux': {'vmin': -120, 'vmax': 0, 'cmap': 'Blues_r'},
                         'mean_surface_net_short_wave_radiation_flux': {'vmin': 0, 'vmax': 350, 'cmap': 'Reds'},
                        'mean_surface_downward_long_wave_radiation_flux': {'vmin': -120, 'vmax': 120, 'cmap': 'RdBu_r'},
                         'mean_surface_downward_short_wave_radiation_flux': {'vmin': -120, 'vmax': 120, 'cmap': 'RdBu_r'},
                         'sea_ice_extent': {'vmin': 0, 'vmax': 400, 'cmap': 'viridis'},
                        'sea_ice_fraction': {'vmin': 0, 'vmax': 1, 'cmap': 'viridis'},
                        'instantaneous_eastward_turbulent_surface_stress': {'vmin': -0.3, 'vmax': 0.3, 'cmap': 'RdBu_r'},
                        'instantaneous_northward_turbulent_surface_stress': {'vmin': -0.3, 'vmax': 0.3, 'cmap': 'RdBu_r'}}


with open(os.path.join(ece3_spinup_dir, f'time_mean_state_dict.pkl'), 'rb') as ifh:
    ece_spinup_tmean_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_40yr_eval_dir, f'time_mean_state_dict.pkl'), 'rb') as ifh:
    ace2_nemo_40yr_eval_tmean_dict = pickle.load(ifh)

with open(os.path.join(ace2_nemo_70yr_eval_dir, f'time_mean_state_dict.pkl'), 'rb') as ifh:
    ace2_nemo_70yr_eval_tmean_dict = pickle.load(ifh)
    
with open(os.path.join(ace2_nemo_spinup_dir, f'time_mean_state_dict.pkl'), 'rb') as ifh:
    ace2_nemo_spinup_tmean_dict = pickle.load(ifh)

In [ ]:
plot_vars= [
           'sea_surface_temperature', 
            'sea_surface_height',
            'sea_surface_salinity'
            ]

time_period='All'

width_height_ratio = [8,6]
shrink_factor= 1
wspace=0.001
cbar_height_ratio=0.02
lat_ticks=None
lon_ticks=None
projection=ccrs.Robinson()



ece3_ds = ece_spinup_tmean_dict[time_period].transpose('latitude', 'longitude', ...)
ace2_nemo_ds = ace2_nemo_40yr_eval_tmean_dict[time_period].transpose('latitude', 'longitude', ...)
da_grid = [ [ece3_ds[varname],  
             ace2_nemo_ds[varname],
            ace2_nemo_ds[varname] -ece3_ds[varname]] for varname in plot_vars]


cbar_labels= [f"{name_lookup[varname]['name']} mean [{name_lookup[varname]['units']}]" for varname in plot_vars]
titles_grid = [['a) Baseline', 'b) ACE2-NEMO-eval', 'c) Difference'], 
               ['d Baseline', 'e) ACE2-NEMO-eval', 'f) Difference'],
              ['g) Baseline', 'h) ACE2-NEMO-eval', 'i) Difference']]

vmax_grid = [[305, 305, 5], [5.0, 5.0, 0.5], [50, 50, 2]]
vmin_grid = [[270, 270, -5], [-2.0,-2.0, -0.5], [0.0, 0.0, -2]]

cmaps = [mean_state_range_dict.get(varname, {}).get('cmap', 'RdBu_r') for varname in plot_vars]
num_rows = len(da_grid)
num_cols = len(da_grid[0])
    
fig = plt.figure(constrained_layout=True, figsize=(num_cols*shrink_factor*width_height_ratio[0], num_rows*shrink_factor*width_height_ratio[1]))
    
gs = gridspec.GridSpec(num_rows * 2, 2*num_cols, figure=fig, 
                    width_ratios=[1]* 2*num_cols,
                    height_ratios=[1, 0.02] * num_rows,
                       wspace=wspace) 
plot_axs = [[fig.add_subplot(gs[2*m, 2*n:2*n+2], projection = projection) for n in range(num_cols)] for m in range(num_rows)]
    
    
for row in range(num_rows):
    for col in range(num_cols):
        
        plot_da = da_grid[row][col]

        im = plot_da.plot(ax=plot_axs[row][col], 
                          vmax=vmax_grid[row][col], 
                          vmin=vmin_grid[row][col], 
                          cmap=cmaps[row], 
                          add_colorbar=False, rasterized=True,
                          transform=ccrs.PlateCarree())
        if col == 1:
            im1=im
        elif col==2:
            im2=im

        plot_axs[row][col].set_title(titles_grid[row][col])
        plot_axs[row][col].coastlines()

    cbar_ax1 = fig.add_subplot(gs[2*row+1, 1:3])
    cbar1 = plt.colorbar(im1, cax=cbar_ax1, label=cbar_labels[row], orientation='horizontal')
    cbar1.ax.tick_params(labelsize=10)

    cbar_ax2 = fig.add_subplot(gs[2*row+1, 4:])
    cbar2 = plt.colorbar(im2, cax=cbar_ax2, label=f"{name_lookup[plot_vars[row]]['abbrev']} mean [{name_lookup[plot_vars[row]]['units']}]", orientation='horizontal')
    cbar2.ax.tick_params(labelsize=10)
    
# plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f"mean_state_{time_period}.pdf"), format='pdf', bbox_inches='tight')

In [ ]:
# Plot of differences in mean state for the eval period
plot_vars= [
           'sea_surface_temperature', 
            'sea_surface_height',
            'sea_surface_salinity'
            ]
fig, axs = plt.subplots(1, len(plot_vars), subplot_kw={'projection': ccrs.Robinson()}, figsize=(len(plot_vars)*8, 5))

time_period = 'last 20 years'
ece3_ds = ece_spinup_tmean_dict[time_period].transpose('latitude', 'longitude', ...)
ace2_nemo_ds = ace2_nemo_40yr_eval_tmean_dict[time_period].transpose('latitude', 'longitude', ...)
da_list = [ ace2_nemo_ds[varname] -ece3_ds[varname] for varname in plot_vars]

vmax_vals = [2, 0.4, 1.0]
vmin_vals = [-2, -0.4, -1.0]
for n, da in enumerate(da_list):

    im = da.plot(ax=axs[n], transform=ccrs.PlateCarree(),
           add_colorbar=False,
           vmin=vmin_vals[n],
           vmax=vmax_vals[n],
           rasterized=True,
                cmap='RdBu_r')
    axs[n].set_title(f"({string.ascii_lowercase[n]}) {name_lookup[plot_vars[n]]['name']} difference\n {time_period.title()}")
    plt.colorbar(im,
                 ax=axs[n], 
                 label=f"{name_lookup[plot_vars[n]]['abbrev']} difference [{name_lookup[plot_vars[n]]['units']}]",
                 orientation='horizontal',
                fraction=0.05)
    axs[n].coastlines()
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f"mean_state_diffs_eval_{time_period.replace(' ', '-')}.pdf"), format='pdf', bbox_inches='tight')

In [ ]:
# Plot of differences in mean state for the spinup period
plot_vars= [
           'sea_surface_temperature', 
            'sea_surface_height',
            'sea_surface_salinity'
            ]
fig, axs = plt.subplots(1, len(plot_vars), subplot_kw={'projection': ccrs.Robinson()}, figsize=(len(plot_vars)*8, 5))

time_period = '1970-1990'
ece3_ds = ece_spinup_tmean_dict[time_period].transpose('latitude', 'longitude', ...)
ace2_nemo_ds = ace2_nemo_spinup_tmean_dict[time_period].transpose('latitude', 'longitude', ...)
da_list = [ ace2_nemo_ds[varname] -ece3_ds[varname] for varname in plot_vars]

vmax_vals = [5, 1, 6.0]
vmin_vals = [-5, -1, -6.0]
for n, da in enumerate(da_list):

    im = da.plot(ax=axs[n], transform=ccrs.PlateCarree(),
           add_colorbar=False,
           vmin=vmin_vals[n],
           vmax=vmax_vals[n],
           rasterized=True,
                cmap='RdBu_r')
    axs[n].set_title(f"({string.ascii_lowercase[n]}) {name_lookup[plot_vars[n]]['name']} difference\n {time_period.title()}")
    plt.colorbar(im,
                 ax=axs[n], 
                 label=f"{name_lookup[plot_vars[n]]['abbrev']} difference [{name_lookup[plot_vars[n]]['units']}]",
                 orientation='horizontal',
                fraction=0.05)
    axs[n].coastlines()

plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f"mean_state_diffs_spinup_{time_period}.pdf"), format='pdf', bbox_inches='tight')

In [ ]:
time_period ='1980-1990'
num_rows = 2
num_cols = 4
width_height_ratio = [6,6]
shrink_factor = 0.7
wspace=0.05
sea_ice_mask = ~np.isnan(ace2_nemo_spinup_tmean_dict[time_period]['sea_ice_fraction'])

land_50m = cfeature.NaturalEarthFeature('physical', 'land', '50m',
                                        edgecolor=cfeature.COLORS['land'],
                                        facecolor=cfeature.COLORS['land'])

satellite_height = 2000000

fig = plt.figure(constrained_layout=True, figsize=(shrink_factor*width_height_ratio[0]*num_cols, shrink_factor*width_height_ratio[1]*num_rows))

gs = gridspec.GridSpec(num_rows + 1, num_cols, figure=fig, 
                    width_ratios=[1]* num_cols,
                    height_ratios=[1] * num_rows + [0.1],
                       wspace=wspace)

projections = [ccrs.NearsidePerspective(central_longitude=-140.0, 
                                                         central_latitude=90,
                                                         false_easting=0,
                                                         satellite_height=satellite_height),
              ccrs.NearsidePerspective(central_longitude=-140.0, 
                                                         central_latitude=-90,
                                                         false_easting=0,
                                                         satellite_height=satellite_height)]

plot_axs = [[fig.add_subplot(gs[m, 0:2], projection = projections[m]), fig.add_subplot(gs[m, 1:3], projection = projections[m])]
            for m in range(num_rows)]


da_list = [ece_spinup_tmean_dict[time_period]['sea_ice_fraction'].transpose('latitude','longitude'),
              ace2_nemo_spinup_tmean_dict[time_period]['sea_ice_fraction'].transpose('latitude','longitude')]

titles_grid = [[f'a) Baseline ({time_period.title()})', f'b) ACE2-NEMO-spinup ({time_period.title()})'], [f'c) Baseline ({time_period.title()})', f'd) ACE2-NEMO-spinup ({time_period.title()})']]

for row in range(num_rows):
    for col in range(2):

        im = da_list[col].plot(ax=plot_axs[row][col], 
                          vmax=1, vmin=0, 
                          cmap='viridis', 
                          add_colorbar=False, rasterized=True,
                          transform=ccrs.PlateCarree())
        plot_axs[row][col].coastlines()
        plot_axs[row][col].add_feature(land_50m)

        plot_axs[row][col].set_title(titles_grid[row][col])

cbar_ax = fig.add_subplot(gs[row+1, 1:2])
cbar = plt.colorbar(im, cax=cbar_ax, 
                    label='Sea ice fraction [0-1]', 
                    orientation='horizontal',
                    fraction=0.05)
cbar.ax.tick_params(labelsize=10)
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f"mean_state_spinup_ice_{time_period}.pdf"), format='pdf', bbox_inches='tight')

In [ ]:
# Same as above but now for the eval periods

time_period ='last 20 years'
num_rows = 2
num_cols = 4
width_height_ratio = [6,6]
shrink_factor = 0.7
wspace=0.05
sea_ice_mask = ~np.isnan(ace2_nemo_40yr_eval_tmean_dict[time_period]['sea_ice_fraction'])

land_50m = cfeature.NaturalEarthFeature('physical', 'land', '50m',
                                        edgecolor=cfeature.COLORS['land'],
                                        facecolor=cfeature.COLORS['land'])

satellite_height = 2000000

fig = plt.figure(constrained_layout=True, figsize=(shrink_factor*width_height_ratio[0]*num_cols, shrink_factor*width_height_ratio[1]*num_rows))

gs = gridspec.GridSpec(num_rows + 1, num_cols, figure=fig, 
                    width_ratios=[1]* num_cols,
                    height_ratios=[1] * num_rows + [0.1],
                       wspace=wspace)

projections = [ccrs.NearsidePerspective(central_longitude=-140.0, 
                                                         central_latitude=90,
                                                         false_easting=0,
                                                         satellite_height=satellite_height),
              ccrs.NearsidePerspective(central_longitude=-140.0, 
                                                         central_latitude=-90,
                                                         false_easting=0,
                                                         satellite_height=satellite_height)]

plot_axs = [[fig.add_subplot(gs[m, 0:2], projection = projections[m]), fig.add_subplot(gs[m, 1:3], projection = projections[m])]
            for m in range(num_rows)]


da_list = [ece_spinup_tmean_dict[time_period]['sea_ice_fraction'].transpose('latitude','longitude'),
              ace2_nemo_40yr_eval_tmean_dict[time_period]['sea_ice_fraction'].transpose('latitude','longitude')]

titles_grid = [[f'a) Baseline ({time_period.title()})', f'b) ACE2-NEMO-eval ({time_period.title()})'], [f'c) Baseline ({time_period.title()})', f'd) ACE2-NEMO-eval ({time_period.title()})']]

for row in range(num_rows):
    for col in range(2):

        im = da_list[col].plot(ax=plot_axs[row][col], 
                          vmax=1, vmin=0, 
                          cmap='viridis', 
                          add_colorbar=False, rasterized=True,
                          transform=ccrs.PlateCarree())
        plot_axs[row][col].coastlines()
        plot_axs[row][col].add_feature(land_50m)

        plot_axs[row][col].set_title(titles_grid[row][col])

cbar_ax = fig.add_subplot(gs[row+1, 1:2])
cbar = plt.colorbar(im, cax=cbar_ax, 
                    label='Sea ice fraction [0-1]', 
                    orientation='horizontal',
                   fraction=0.05)
cbar.ax.tick_params(labelsize=10)
plt.savefig(os.path.join(MANUSCRIPT_FIGURE_DIR, f"mean_state_eval_ice_{time_period}.pdf"), format='pdf', bbox_inches='tight')